In [ ]:
from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import pandas as pd

columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv("data.csv", encoding='latin1', names=columns)
df['target'] = df['target'].replace(4, 1)

In [ ]:
df_small = df.sample(n=10000, random_state=42)

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

df_small['clean_text'] = df_small['text'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_small['clean_text']
y = df_small['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)
test_dataset = Dataset(test_encodings, y_test)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=50
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
import numpy as np

y_pred = np.argmax(predictions.predictions, axis=1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.save_pretrained('/content/drive/MyDrive/depression_model')
tokenizer.save_pretrained('/content/drive/MyDrive/depression_model')

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_pred
})

results_df.to_csv('/content/drive/MyDrive/results.csv', index=False)

In [ ]:
df_small.to_csv('/content/drive/MyDrive/df_small.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import re

In [ ]:
columns = ['target', 'id', 'date', 'flag', 'user', 'text']

df = pd.read_csv("data.csv", encoding='latin1', names=columns)
df['target'] = df['target'].replace(4, 1)

df_small = df.sample(n=10000, random_state=42)

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

df_small['clean_text'] = df_small['text'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_small['clean_text']
y = df_small['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

y_pred_lr = model_lr.predict(X_test_tfidf)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
df.to_csv('/content/drive/MyDrive/data.csv', index=False)

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred_lr': y_pred_lr
})

results_df.to_csv('/content/drive/MyDrive/baseline_results.csv', index=False)

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/data.csv")

### Experiment 2: Training with 50K Samples

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd

columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv("data.csv", encoding='latin1', names=columns)

In [ ]:
df['target'] = df['target'].replace(4, 1)
df['target'] = df['target'].astype(int)

print(df['target'].unique())

In [ ]:
df_small = df.sample(n=50000, random_state=42)

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

df_small['clean_text'] = df_small['text'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_small['clean_text']
y = df_small['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_lr = LogisticRegression(max_iter=200)
model_lr.fit(X_train_tfidf, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_lr = model_lr.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_probs_lr = model_lr.predict_proba(X_test_tfidf)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_probs_lr)
auc_lr = auc(fpr_lr, tpr_lr)

plt.plot(fpr_lr, tpr_lr, label=f"LR AUC = {auc_lr:.2f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()

In [ ]:
!pip install transformers datasets

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.to_csv('/content/drive/MyDrive/data.csv', index=False)

In [ ]:
df_small.to_csv('/content/drive/MyDrive/data_50k.csv', index=False)

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
import pandas as pd

df_small = pd.read_csv('/content/drive/MyDrive/data_50k.csv')

In [ ]:
X = df_small['text'].astype(str)
y = df_small['target']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(X_train), truncation=True, padding=True)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(y_train)
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(y_test)
})

In [ ]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=500,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"BERT AUC = {roc_auc:.2f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve - BERT")
plt.show()

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "BERT"],
    "Accuracy": [0.7725, 0.833],
    "AUC": [0.85, 0.91]
})

results.to_csv('/content/drive/MyDrive/final_results.csv', index=False)

In [ ]:
plt.savefig('/content/drive/MyDrive/lr_roc.png')

In [ ]:
plt.savefig('/content/drive/MyDrive/bert_roc.png')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr)
plt.savefig('/content/drive/MyDrive/lr_cm.png')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.savefig('/content/drive/MyDrive/bert_cm.png')

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
from google.colab import files
files.upload()

In [1]:
import os
print(os.listdir('/content/drive/MyDrive'))

['IIIT_Vadodara_Resume (1).pdf', 'Colab Notebooks', 'depression_model', 'data.csv', 'results.csv', 'df_small.csv', 'baseline_results.csv', 'final_results.csv', 'lr_roc.png', 'bert_roc.png', 'lr_cm.png', 'bert_cm.png', 'data_50k.csv']
